# Import Packages

I'll make use of the following packages:
- `numpy` is a package for scientific computing in python.
- `pandas` A powerful Python library for data manipulation and analysis.
- `seaborn` A data visualization library based on matplotlib.
- `scikit-learn` A comprehensive library for machine learning in Python.
- `kaggle` Using Kaggle API to download data.

In [477]:
import numpy as np
import pandas as pd
import kagglehub
import os
import re
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict


# Download Data

In [478]:
# Download latest version
path = kagglehub.dataset_download("taeefnajib/used-car-price-prediction-dataset")

print(os.listdir(path))

['used_cars.csv']


In [479]:
# Load the dataset
df = pd.read_csv(os.path.join(path, "used_cars.csv"))

# Data preprocessing

## Check data

In [480]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [481]:
## check for missing values
df.isna().sum()

brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64

## 📝 Initial Data Glance – Observations & Next Steps

### Overview

- **Columns:**  
  The dataset contains **12 columns**: `brand`, `model`, `model_year`, `milage`, `fuel_type`, `engine`, `transmission`, `ext_col`, `int_col`, `accident`, `clean_title`, `price`.

- **Data Types:**  
  Most columns are **object type** (categorical or text). Data transformation will be required (label encoding, one-hot encoding, parsing text fields).

- **Brand & Model:**  
  - Some `model` values are duplicated or concatenated (e.g., `RX 350 RX 350`).
  - Will inspect and **clean/split/merge brand and model** to ensure unique, consistent values.

- **Feature Extraction:**  
  - Columns such as `engine` and `transmission` contain multiple details (e.g., horsepower, engine size, cylinder count, speed type) that can be **parsed into new features**.

- **Missing Values:**  
  - Nulls detected in several columns (e.g., `clean_title`).  
  - Will analyze missingness and apply appropriate imputation (mode, new category, or predictive imputation).

- **Target Variable:**  
  - The `price` column is a string with currency symbol and commas—needs to be cleaned and converted to numeric.

- **Formatting Issues:**  
  - Fields like `milage` and `price` contain units/symbols (e.g., "mi.", "$", ",")—will remove for numeric conversion.
  - `accident` and `clean_title` are categorical but may need binarization or mapping.

---

### Next Steps

- Clean and standardize all categorical and text fields.
- Parse and extract features from `engine` and `transmission` columns.
- Handle missing values with suitable imputation strategies.
- Convert `price` and `milage` to numeric types.
- Ensure brand/model consistency for analysis and modeling.

## Column Transformation

### Brand and Model

In [482]:
brand_list = sorted(df['brand'].unique())

by_letter = defaultdict(list)
for brand in brand_list:
    by_letter[brand[0].upper()].append(brand)
    

for letter in by_letter.keys():
    print(f"{letter}:{', '.join(by_letter[letter])}")

A:Acura, Alfa, Aston, Audi
B:BMW, Bentley, Bugatti, Buick
C:Cadillac, Chevrolet, Chrysler
D:Dodge
F:FIAT, Ferrari, Ford
G:GMC, Genesis
H:Honda, Hummer, Hyundai
I:INFINITI
J:Jaguar, Jeep
K:Karma, Kia
L:Lamborghini, Land, Lexus, Lincoln, Lotus, Lucid
M:MINI, Maserati, Maybach, Mazda, McLaren, Mercedes-Benz, Mercury, Mitsubishi
N:Nissan
P:Plymouth, Polestar, Pontiac, Porsche
R:RAM, Rivian, Rolls-Royce
S:Saab, Saturn, Scion, Subaru, Suzuki, smart
T:Tesla, Toyota
V:Volkswagen, Volvo


---
🏷️ Brand Name Inconsistencies

While reviewing the car brands, I noticed that several are missing their full names:

- **Alfa** → _Alfa Romeo_
- **Aston** → _Aston Martin_
- **Land** → _Land Rover_

Let’s take a closer look at these brands to ensure correct and consistent naming throughout the dataset.

In [483]:
df[df['brand'] == 'Alfa'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
151,Alfa,Romeo Stelvio Ti Sport,2020,"18,665 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Lunare White Metallic,Ice,None reported,Yes,"$35,645"
255,Alfa,Romeo Giulia Quadrifoglio,2022,"1,966 mi.",Gasoline,2.9L V6 24V GDI DOHC Twin Turbo,8-Speed Automatic,Verde,Black,None reported,NaN,"$75,900"
343,Alfa,Romeo Stelvio Ti,2020,"41,000 mi.",Gasoline,280.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$32,400"
412,Alfa,Romeo Stelvio Quadrifoglio,2019,"26,500 mi.",Gasoline,505.0HP 2.9L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Gray,Black,None reported,Yes,"$53,900"
414,Alfa,Romeo Stelvio Ti Sport,2020,"21,487 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Anodized Blue Metallic,Ice,None reported,Yes,"$35,345"


In [484]:
df[df['brand'] == 'Aston'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
11,Aston,Martin DBS Superleggera,2019,"22,770 mi.",Gasoline,715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes,"$184,606"
93,Aston,Martin DBS Superleggera,2021,"2,165 mi.",Gasoline,5.2L V12 48V GDI DOHC Twin Turbo,8-Speed Automatic,Black,Black,None reported,Yes,"$279,950"
314,Aston,Martin DBX Base,2021,"2,353 mi.",Gasoline,4.0L V8 32V GDI DOHC Twin Turbo,9-Speed Automatic,Green,Sahara Tan,None reported,Yes,"$159,500"
535,Aston,Martin V8 Vantage Base,2008,"25,025 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,"$39,000"
610,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"


In [485]:
df[df['brand'] == 'Land'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
10,Land,Rover Range Rover Sport 3.0 Supercharged HST,2021,"27,608 mi.",Gasoline,V6,Automatic,Fuji White,Pimento / Ebony,None reported,NaN,"$73,897"
15,Land,Rover LR4 HSE,2013,"79,800 mi.",Gasoline,375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,White,Black,None reported,Yes,"$29,990"
80,Land,Rover Discovery Sport SE R-Dynamic,2020,"21,240 mi.",Gasoline,2.0 Liter,Automatic,White,Black,None reported,NaN,"$37,998"
110,Land,Rover LR4 HSE LUX Landmark Edition,2016,"144,000 mi.",Gasoline,340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$18,000"
120,Land,Rover Range Rover Sport 3.0L Supercharged HSE,2018,"104,700 mi.",Gasoline,V6,Automatic,Fuji White,Ivory / Ebony,At least 1 accident or damage reported,NaN,"$30,775"


---
### 🔧 Brand Name Corrections Needed

As predicted, these three brands require their brand and model names to be fixed for consistency:

- **Alfa** → _Alfa Romeo_
- **Aston** → _Aston Martin_
- **Land** → _Land Rover_

In [486]:
### Fixing Brand Name Inconsistencies

df_update = df.copy()

## Replace inconsistent brand names with full names
df_update['brand'] = df_update['brand'].replace({
    'Alfa': 'Alfa Romeo',
    'Aston': 'Aston Martin',
    'Land': 'Land Rover'
})

## Remove first word from model names for these brands
brand_name = ['Alfa Romeo', 'Aston Martin', 'Land Rover']

for brand in brand_name:

    df_update.loc[df_update['brand'] == brand, 'model'] = df_update.loc[df_update['brand'] == brand, 'model'].str.split().apply(lambda x: ' '.join(x[1:]) if isinstance(x, list) and len(x) > 1 else '')


In [487]:
df_update[df_update['brand'] == 'Alfa Romeo'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
151,Alfa Romeo,Stelvio Ti Sport,2020,"18,665 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Lunare White Metallic,Ice,None reported,Yes,"$35,645"
255,Alfa Romeo,Giulia Quadrifoglio,2022,"1,966 mi.",Gasoline,2.9L V6 24V GDI DOHC Twin Turbo,8-Speed Automatic,Verde,Black,None reported,NaN,"$75,900"
343,Alfa Romeo,Stelvio Ti,2020,"41,000 mi.",Gasoline,280.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,8-Speed A/T,White,Black,None reported,Yes,"$32,400"
412,Alfa Romeo,Stelvio Quadrifoglio,2019,"26,500 mi.",Gasoline,505.0HP 2.9L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Gray,Black,None reported,Yes,"$53,900"
414,Alfa Romeo,Stelvio Ti Sport,2020,"21,487 mi.",Gasoline,2.0L I4 16V GDI SOHC Turbo,8-Speed Automatic,Anodized Blue Metallic,Ice,None reported,Yes,"$35,345"


In [488]:
df_update[df_update['brand'] == 'Aston Martin'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
11,Aston Martin,DBS Superleggera,2019,"22,770 mi.",Gasoline,715.0HP 5.2L 12 Cylinder Engine Gasoline Fuel,8-Speed A/T,Silver,Black,None reported,Yes,"$184,606"
93,Aston Martin,DBS Superleggera,2021,"2,165 mi.",Gasoline,5.2L V12 48V GDI DOHC Twin Turbo,8-Speed Automatic,Black,Black,None reported,Yes,"$279,950"
314,Aston Martin,DBX Base,2021,"2,353 mi.",Gasoline,4.0L V8 32V GDI DOHC Twin Turbo,9-Speed Automatic,Green,Sahara Tan,None reported,Yes,"$159,500"
535,Aston Martin,V8 Vantage Base,2008,"25,025 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,6-Speed A/T,White,Black,At least 1 accident or damage reported,Yes,"$39,000"
610,Aston Martin,V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"


In [489]:
df_update[df_update['brand'] == 'Land Rover'].head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
10,Land Rover,Range Rover Sport 3.0 Supercharged HST,2021,"27,608 mi.",Gasoline,V6,Automatic,Fuji White,Pimento / Ebony,None reported,NaN,"$73,897"
15,Land Rover,LR4 HSE,2013,"79,800 mi.",Gasoline,375.0HP 5.0L 8 Cylinder Engine Gasoline Fuel,A/T,White,Black,None reported,Yes,"$29,990"
80,Land Rover,Discovery Sport SE R-Dynamic,2020,"21,240 mi.",Gasoline,2.0 Liter,Automatic,White,Black,None reported,NaN,"$37,998"
110,Land Rover,LR4 HSE LUX Landmark Edition,2016,"144,000 mi.",Gasoline,340.0HP 3.0L V6 Cylinder Engine Gasoline Fuel,8-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$18,000"
120,Land Rover,Range Rover Sport 3.0L Supercharged HSE,2018,"104,700 mi.",Gasoline,V6,Automatic,Fuji White,Ivory / Ebony,At least 1 accident or damage reported,NaN,"$30,775"


---

Now that I’ve fixed the brand names, let’s address another issue: **duplicate model names**.  
For example, in row 2 I saw `Lexus RX 350 RX 350` as a model. These duplicates inflate the number of unique model groups.

**Next step:**  
Clean the `model` column to remove repeated names and reduce redundancy in our model grouping.

### 🔧 Clean Model names

In [490]:
# Remove duplicate model names
# For example, 'RX 350 RX 350' should be 'RX 350'
df_update['model'] = df_update['model'].apply(lambda x: ' '.join(dict.fromkeys(x.split())))

In [491]:
## particularly for BMW models, clean up mutant model names with duplicated numeric badges
def clean_bmw_model(model):
    """
    Cleans up mutant BMW model names with duplicated numeric badges.
    Examples:
        '330 330i xDrive'         -> '330 i xDrive'
        'M550 M550i xDrive'       -> 'M550 i xDrive'
        '440 Gran Coupe 440i xDrive' -> '440 Gran Coupe i xDrive'
        '428 Gran Coupe 428i xDrive SULEV' -> '428 Gran Coupe i xDrive SULEV'
    Only works if the second badge starts with the first (e.g., '330' in '330i').
    """
    tokens = str(model).split()
    if len(tokens) < 2:
        return model
    for i in range(1, len(tokens)):
        if tokens[i].startswith(tokens[0]) and tokens[i].endswith('i'):
            suffix = tokens[i][len(tokens[0]):]  # Should just be 'i'
            tokens = tokens[:i] + [suffix] + tokens[i+1:]
            # Clean up empty tokens and extra spaces, just in case
            return ' '.join([t for t in tokens if t])
    return model

In [492]:
## Apply the cleaning function to BMW models
mask = df_update['brand'] == 'BMW'
df_update.loc[mask, 'model'] = df_update.loc[mask, 'model'].apply(clean_bmw_model)


### Extract numbers columns

In [493]:
col_names = ['milage', 'price']

def clean_numeric_col(series):
    """
    Clean numeric columns by removing non-numeric characters and converting to float.
    """
    return (
        series.astype(str)  # Ensure the series is of string type
        .str.replace(r'[^\d.]', '', regex=True)  # Remove non-numeric characters except digits and "."
        .astype(float)  # Convert to float
    )

# Apply the cleaning function to the specified columns
for col in col_names:
    df_update[col] = clean_numeric_col(df[col])

🔢 Now, both `milage` and `price` have been successfully converted to numerical columns.

---

### Individual columns
#### ⛽ Fuel Type: Data Cleaning Needed

In [494]:
df_update['fuel_type'].unique()

array(['E85 Flex Fuel', 'Gasoline', 'Hybrid', nan, 'Diesel',
       'Plug-In Hybrid', '–', 'not supported'], dtype=object)

The `fuel_type` column contains multiple categories, null values, and some unsupported or placeholder entries (e.g., `nan`, `'–'`, `'not supported'`).  
I’ll need to clean and unify these values for consistent analysis and modeling.

In [495]:
mask = df_update['fuel_type'].isin(['–', 'not supported']) | df_update['brand'].isna()
df.loc[mask, ['brand', 'model', 'fuel_type']]['brand'].value_counts()

brand
Dodge            8
Toyota           5
Ford             5
Mazda            4
Chrysler         3
Chevrolet        3
Cadillac         3
Nissan           3
Porsche          2
Acura            2
Mercedes-Benz    2
Rolls-Royce      1
Mercury          1
Volvo            1
Jaguar           1
Jeep             1
Honda            1
GMC              1
Name: count, dtype: int64

Tesla, Lucid, and Rivian are pure electric brands—missing `fuel_type` values for these can be safely filled as `'Electric'`.  
For all other brands, I’ll need to inspect the model before imputing the correct fuel type.

In [496]:
mask = (df_update['brand'].isin(['Tesla', 'Lucid', 'Rivian']) & 
        (df_update['fuel_type'].isin(['–', 'not supported']) |
         df_update['fuel_type'].isna()))

# Fill missing fuel_type for electric brands
df_update.loc[mask, 'fuel_type'] = 'Electric'

In [497]:
## Check if same models already have fuel_type
mask = (~df_update['fuel_type'].isin(['–', 'not supported'])) & (~df_update['fuel_type'].isna())
fuel_type_map = (
    df_update[mask].groupby(['brand', 'model'])['fuel_type']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)
    .to_dict()
)

In [498]:
## Impute fuel_type

def impute_fuel_type(row):
    """
    Impute fuel_type based on brand and model.

    Args:
        row (pd.Series): A row of the DataFrame.
        
    Returns:
        str: The imputed transmission speeds.
    """
    mask = (row['fuel_type'] in ['–', 'not supported']) | (pd.isna(row['fuel_type']))
    if mask:
        key = (row['brand'], row['model'])
        return fuel_type_map.get(key, row['fuel_type'])
    else:
        return row['fuel_type']
    
## Apply the imputation function to the DataFrame
df_update['fuel_type'] = df_update.apply(impute_fuel_type, axis=1)


In [499]:
# update fuel_type for other brands that's not available in the fuel_type_map

# Mapping dictionary: brand, model -> fuel type
map_fuel_type = {
    # Hydrogen
    ('Toyota', 'Mirai Base'): 'Hydrogen',
    ('Toyota', 'Mirai Limited'): 'Hydrogen',
    # Plug-In Hybrid/Range Extender
    ('BMW', 'i3 Base w/Range Extender'): 'Plug-In Hybrid',
    ('Karma', 'Revero Base'): 'Plug-In Hybrid',
    # Diesel
    ('Mercedes-Benz', 'E-Class D 2.5 Turbo'): 'Diesel'
}

def assign_fuel_type(row):
    if pd.isna(row['fuel_type']) or row['fuel_type'] in ['–', 'not supported']:
        key = (row['brand'], row['model'])

        return map_fuel_type.get(key, row['fuel_type'])
    else:
        return row['fuel_type']
    
# Apply the mapping to the DataFrame
df_update['fuel_type'] = df_update.apply(assign_fuel_type, axis=1)

## Update fuel_type dictionary 
fuel_type_map.update(map_fuel_type)

In [500]:
# Save the new transmission type mapping
with open('fuel_type_map.pkl', 'wb') as f:
    pickle.dump(fuel_type_map, f)

# ## Load it back
# with open('fuel_type_map.pkl', 'rb') as f:
#     fuel_type_map = pickle.load(f)

✅ Fuel Type Cleanup Complete

All missing and inconsistent `fuel_type` values have been handled.  
The column is now clean and ready for analysis and modeling.

---

#### Engine

In [501]:
sorted(df_update['engine'].unique())

['1.2L I3 12V GDI DOHC Turbo',
 '1.3L I3 12V GDI DOHC Turbo',
 '1.3L I3 12V MPFI DOHC Turbo',
 '1.4L I4 16V GDI DOHC Turbo',
 '1.5 Liter Turbo',
 '1.5L I3 12V GDI DOHC Turbo',
 '1.5L I3 12V PDI DOHC Turbo',
 '1.5L I4 16V GDI DOHC Turbo',
 '1.6L I-4 gasoline direct injection, DOHC, variable valve control',
 '1.6L I4 16V GDI DOHC',
 '1.6L I4 16V GDI DOHC Hybrid',
 '1.6L I4 16V GDI DOHC Turbo',
 '1.6L I4 16V GDI DOHC Turbo Hybrid',
 '1.6L I4 16V MPFI DOHC',
 '1.8 Liter',
 '101.0HP 1.4L 4 Cylinder Engine Gasoline Fuel',
 '1020.0HP Electric Motor Electric Fuel System',
 '104.0HP 1.6L 4 Cylinder Engine Gasoline Fuel',
 '106.0HP 1.5L 4 Cylinder Engine Gasoline Fuel',
 '107.0HP Electric Motor Electric Fuel System',
 '109.0HP 1.5L 4 Cylinder Engine Gasoline Fuel',
 '109.0HP 1.6L 4 Cylinder Engine Gasoline Fuel',
 '111.0HP Electric Motor Electric Fuel System',
 '111.2Ah / FR 70kW / RR 160kW (697V)',
 '115.0HP 1.6L 4 Cylinder Engine Gasoline Fuel',
 '115.0HP 1.7L 4 Cylinder Engine Gasoline Fuel',

---

#### ⚙️ Transmission Data: Standardization & Feature Extraction

The `transmission` column contains various types and naming conventions for similar transmissions.  
To ensure consistency and improve analysis, we’ll standardize these values and extract two separate features:
- **Transmission Type** (e.g., Automatic, Manual, CVT)
- **Number of Speeds** (e.g., 6, 8, Single-Speed)

In [502]:
## Identify and Standardize Transmission Types
"""
This step will focus on standardizing the values with both speed and mannual or automatic transmission.
Two new columns will be created:
- **Transmission Type** (e.g., Automatic, Manual, CVT)
- **Number of Speeds** (e.g., 6, 8, Single-Speed
"""

def extract_transmission_type(val):
    """
    Extracts the transmission type from the given value.

    Args:
        val (str): The transmission value to extract from.

    Returns:
        str: The transmission type ('Automatic', 'Manual', 'CVT', or 'Other').
    """
    v = str(val).lower()
    if any(keyword in v for keyword in ['automatic', 'a/t', 'auto', 'dual-clutch',
                                         'steptronic', 'dct', 'pdk', 'at', 'dual shift mode', 
                                         'overdrive switch', 'single-speed']):
        return 'Automatic'
    if any(keyword in v for keyword in ['manual', 'm/t', 'mt']):
        return 'Manual'
    if any(keyword in v for keyword in ['cvt', 'variable']):
        return 'CVT'
    return 'Other'


def extract_transmission_speeds(val):
    """
    Extracts the number of speeds from the given transmission value.

    Args:
        val (str): The transmission value to extract from.

    Returns:
        int: The number of speeds (e.g., 6), or 1 for single-speed, or None if not applicable.
    """
    v = str(val).lower()
    match = re.search(r'(\d+)[-\s]?(?:speed|spd)', v)
    if match:
        return int(match.group(1))
    if 'single-speed' in v or 'single speed' in v:
        return 1
    numbers = re.search(r'(\d+)', v)
    if numbers:
        return int(numbers.group(1))
    return None

💡 **Why Keep “CVT” as Its Own Category?**

1. **Mechanically Different**  
   - **Traditional Automatic:** Uses a set of gears, shifts through them automatically.  
   - **CVT:** No gears—uses pulleys and belts for infinite gear ratios.

2. **User Experience is Different**  
   - CVTs drive differently. No gear shifts, “rubber band” feel.  
   - Impacts consumer reviews, performance, and pricing.

3. **Manufacturer & Industry Reporting**  
   - Specs, consumer guides, and data vendors *always* break out “CVT” separately from “Automatic”.  
   - Lumping them together can hide meaningful patterns (pricing, reliability, satisfaction, etc).

4. **Modeling & Analytics**  
   - Some buyers specifically want to avoid (or seek out) CVTs.  
   - Resale value, repair costs, and reliability trends can differ significantly.  
   - **Keeping “CVT” as its own class maintains transparency and analytic flexibility.**

In [503]:
# df_update.drop(columns=['transmission_type'], inplace=True)

In [504]:
## Apply the Extraction Functions
df_update['transmission_type'] = df_update['transmission'].apply(extract_transmission_type)
df_update['transmission_speeds'] = df_update['transmission'].apply(extract_transmission_speeds)

In [505]:
df_update['transmission_type'].value_counts() 

transmission_type
Automatic    3557
Manual        373
CVT            67
Other          12
Name: count, dtype: int64

In [506]:
df_update['transmission_speeds'].isna().sum()

1832

In [507]:
## check uncategorized transmission types
mask = df_update['transmission_type'] == 'Other'

df_update.loc[mask,['brand', 'model', 'transmission', 'transmission_type']]

,brand,model,transmission,transmission_type
5,Acura,ILX 2.4L,F,Other
269,Acura,TLX w/A-Spec Package,2,Other
476,Acura,MDX w/Technology Package,F,Other
516,Acura,MDX w/Technology Package,2,Other
536,Porsche,911 Carrera S,–,Other
855,Ford,Bronco,–,Other
916,Porsche,911 Carrera 4S,–,Other
1236,Toyota,Tacoma TRD Pro,6-Speed,Other
1356,Lamborghini,Aventador SVJ Base,7-Speed,Other
1615,Rolls-Royce,Phantom,–,Other


As expected, some vehicles have incomplete transmission information.
- **1832 rows** are missing transmission speeds.
- **12 rows** are missing transmission type.

These missing values will need to be reviewed and corrected for accurate analysis.

##### Fix transmission type

In [508]:
## Check if same models already have transmission types

transmission_type_map = (
    df_update[df_update['transmission_type'] != 'Other']
    .groupby(['brand', 'model'])['transmission_type']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else 'Other')
    .to_dict()
)

In [509]:
## Impute transmission_type
def impute_transmission_type(row):
    """
    Impute the transmission type based on the brand and model.
    
    Args:
        row (pd.Series): A row of the DataFrame.
        
    Returns:
        str: The imputed transmission type.
    """
    if row['transmission_type'] == 'Other':
        key = (row['brand'], row['model'])
        return transmission_type_map.get(key, row['transmission_type'])
    return row['transmission_type']

# Apply the imputation function
df_update['transmission_type'] = df_update.apply(impute_transmission_type, axis=1)

In [510]:
## Check for Missing Transmission Types
mask = df_update['transmission_type'] == 'Other'

df_update.loc[mask,['brand', 'model', 'transmission', 'transmission_type']]

,brand,model,transmission,transmission_type
5,Acura,ILX 2.4L,F,Other
269,Acura,TLX w/A-Spec Package,2,Other
476,Acura,MDX w/Technology Package,F,Other
516,Acura,MDX w/Technology Package,2,Other
2381,Acura,RDX PMC Edition,2,Other


In [511]:
## Dictionary mapping (brand, model) to transmission type
map_transmission_type = {
    ('Acura', 'ILX 2.4L'): 'Automatic',
    ('Acura', 'TLX w/A-Spec Package'): 'Automatic',
    ('Acura', 'MDX w/Technology Package'): 'Automatic',
    ('Acura', 'RDX PMC Edition'): 'Automatic'
}

def assign_transmission_type(row):
    """
    Assigns the transmission type based on the brand and model.

    Args:
        row (pd.Series): A row of the DataFrame.

    Returns:
        str: The assigned transmission type.
    """
    if row['transmission_type'] == 'Other':
        key = (row['brand'], row['model'])
        return map_transmission_type.get(key, row['transmission_type'])
    else:
        return row['transmission_type']

## Apply the mapping to the DataFrame
df_update['transmission_type'] = df_update.apply(assign_transmission_type, axis=1)

## Update transmission_type dictionary
transmission_type_map.update(map_transmission_type)





In [512]:
# Save the new transmission type mapping
with open('transmission_type_map.pkl', 'wb') as f:
    pickle.dump(transmission_type_map, f)
    
## Load it back
# with open('transmission_type_map.pkl', 'rb') as f:
#     transmission_type_map = pickle.load(f)

In [513]:
df_update['transmission_type'].value_counts() 

transmission_type
Automatic    3568
Manual        374
CVT            67
Name: count, dtype: int64

✅ Transmission Type Standardization Complete

- All transmission type values have been reviewed and corrected.  
- The column is now ready for analysis and modeling.

---

##### Fix transmission speeds

In [514]:
## Check if same models already have transmission speeds
transmission_speeds_map = (
    df_update[df_update['transmission_speeds'].notna()]
    .groupby(['brand', 'model'])['transmission_speeds']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)
    .to_dict()
)


In [515]:
## Impute transmission_speeds
def impute_transmission_speeds(row):
    """
    Impute the transmission speeds based on the brand and model.
    
    Args:
        row (pd.Series): A row of the DataFrame.
        
    Returns:
        int or None: The imputed transmission speeds.
    """
    if pd.isna(row['transmission_speeds']):
        key = (row['brand'], row['model'])
        return transmission_speeds_map.get(key, row['transmission_speeds'])
    else:
        return row['transmission_speeds']
    
# Apply the imputation function
df_update['transmission_speeds'] = df_update.apply(impute_transmission_speeds, axis=1)

In [516]:
df_update['transmission_speeds'].isna().sum()

918

In [518]:
## Dictionary mapping (brand, model) to transmission speeds
"""
To save space, just display the format of the dictionary.
"""
map_transmission_speeds = {
    # Volvo
    ('Volvo', '850 Turbo'): 4,
    ('Volvo', 'C30 T5 Premier Plus'): 5,
    ('Volvo', 'C40 Recharge Pure Electric Twin Ultimate'): 1,
    ('Volvo', 'S60 B5 Inscription'): 8,
    ('Volvo', 'S60 T5 Premier Plus'): 8,
    ('Volvo', 'S80 3.2'): 6,
    ('Volvo', 'XC60 T5 R-Design'): 8,
    ('Volvo', 'XC60 T6 Inscription'): 8,
    ('Volvo', 'XC70 T6 Platinum'): 6,
    ('Volvo', 'XC90 3.2'): 6,
    ('Volvo', 'XC90 Hybrid T8 R-Design'): 8
}

transmission_speeds_map.update(map_transmission_speeds)

# Save the new transmission type mapping
with open('transmission_speeds_map.pkl', 'wb') as f:
    pickle.dump(transmission_speeds_map, f)

# ## Load it back
# with open('transmission_speeds_map.pkl', 'rb') as f:
#     transmission_speeds_map = pickle.load(f)


In [519]:
## Impute transmission speeds
def assign_transmission_speeds(row):
    """
    Assigns the transmission speeds based on the brand and model.

    Args:
        row (pd.Series): A row of the DataFrame.

    Returns:
        int or None: The assigned transmission speeds.
    """
    if pd.isna(row['transmission_speeds']):
        key = (row['brand'], row['model'])
        return transmission_speeds_map.get(key, row['transmission_speeds'])
    else:
        return row['transmission_speeds']
    
# Apply the mapping to the DataFrame
df_update['transmission_speeds'] = df_update.apply(assign_transmission_speeds, axis=1)

In [520]:
df_update['transmission_speeds'].value_counts()

transmission_speeds
6.0     971
8.0     883
7.0     368
10.0    237
5.0     237
9.0     158
1.0     132
4.0     109
2.0       9
Name: count, dtype: int64

✅ All transmission speeds for colossal list of car models have been assigned and are ready to roll. 


---



In [521]:
mask = ((df_update['model'] == '430 430i'))

df_update.loc[mask,['model']]

,model


In [522]:
mask = ((df_update['brand'] == 'Porsche'))

df_update.loc[mask,['model']].sort_values(by='model').drop_duplicates().values

array([['718 Boxster Base'],
       ['718 Boxster GTS'],
       ['718 Boxster S'],
       ['718 Cayman GT4'],
       ['718 Cayman GTS'],
       ['718 Cayman S'],
       ['718 Spyder Base'],
       ['911 Carrera'],
       ['911 Carrera 4'],
       ['911 Carrera 4 Cabriolet'],
       ['911 Carrera 4 GTS'],
       ['911 Carrera 4S'],
       ['911 Carrera 4S Cabriolet'],
       ['911 Carrera C2S'],
       ['911 Carrera C4S'],
       ['911 Carrera Cabriolet'],
       ['911 Carrera GTS'],
       ['911 Carrera S'],
       ['911 Carrera S Cabriolet'],
       ['911 Carrera Turbo'],
       ['911 GT2 RS'],
       ['911 GT3'],
       ['911 GT3 RS'],
       ['911 R'],
       ['911 Targa 4 GTS'],
       ['911 Turbo'],
       ['911 Turbo Cabriolet'],
       ['911 Turbo S'],
       ['Boxster Base'],
       ['Boxster Black Edition'],
       ['Boxster GTS'],
       ['Boxster RS 60 Spyder'],
       ['Boxster S'],
       ['Carrera GT Base'],
       ['Cayenne AWD'],
       ['Cayenne Base'],
       ['Cayenn

In [523]:
df_update['brand'].value_counts()

brand
Ford             386
BMW              375
Mercedes-Benz    315
Chevrolet        292
Porsche          201
Audi             200
Toyota           199
Lexus            163
Jeep             143
Land Rover       130
Nissan           116
Cadillac         107
GMC               91
RAM               91
Dodge             90
Tesla             87
Kia               76
Hyundai           72
Mazda             64
Acura             64
Subaru            64
Honda             63
Volkswagen        59
INFINITI          59
Lincoln           52
Jaguar            47
Volvo             38
Maserati          34
Bentley           33
MINI              33
Buick             30
Chrysler          28
Lamborghini       26
Mitsubishi        20
Genesis           20
Alfa Romeo        19
Rivian            17
Hummer            16
Pontiac           15
Ferrari           12
Rolls-Royce       11
Aston Martin       9
McLaren            6
Scion              6
FIAT               5
Saturn             5
Lotus              4
Lucid  